In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import Isomap
from sklearn.metrics import classification_report

class ManifoldInferenceBenchmark:
    """
    Экспериментальный пайплайн для оценки эффективности нелинейного
    снижения размерности (Isomap Embeddings) в задачах классификации.
    """
    def __init__(self, n_neighbors_knn: int = 5, n_components_isomap: int = 2):
        self.n_neighbors_knn = n_neighbors_knn
        self.n_components_isomap = n_components_isomap
        self.knn_baseline = KNeighborsClassifier(n_neighbors=n_neighbors_knn)
        self.knn_manifold = KNeighborsClassifier(n_neighbors=n_neighbors_knn)
        self.isomap = Isomap(n_components=n_components_isomap, n_neighbors=5)

    def run_experiment(self):
        # 1. Загрузка валидного датасета (без модификации исходных фич)
        iris = load_iris()
        X, y = iris.data, iris.target

        # 2. Корректное разделение до проведения любых трансформаций (Исключаем Data Leakage)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )

        print(f"[INFO] Исходная размерность пространства признаков: {X_train.shape}")

        # =====================================================================
        # ФАЗА 1: Оценка Baseline-модели на полном пространстве признаков (4D)
        # =====================================================================
        self.knn_baseline.fit(X_train, y_train)
        acc_baseline = self.knn_baseline.score(X_test, y_test)
        print(f"[BENCHMARK] Точность на полном пространстве (4D): {acc_baseline:.4f}")

        # =====================================================================
        # ФАЗА 2: Нелинейное сжатие через Isomap (Геодезическое многообразие)
        # =====================================================================
        print(f"\n[INFO] Старт нелинейного сжатия пространства признаков до {self.n_components_isomap}D...")

        # Обучаем проекцию строго на тренировочном наборе
        X_train_transformed = self.isomap.fit_transform(X_train)
        # Тестовый набор только трансформируем
        X_test_transformed = self.isomap.transform(X_test)

        # Обучение модели на латентном пространстве низкого ранга
        self.knn_manifold.fit(X_train_transformed, y_train)
        acc_manifold = self.knn_manifold.score(X_test_transformed, y_test)

        print(f"[BENCHMARK] Точность после Isomap сжатия ({self.n_components_isomap}D): {acc_manifold:.4f}")

        # =====================================================================
        # ФАЗА 3: Сравнительный анализ распределения ошибок
        # =====================================================================
        y_pred_manifold = self.knn_manifold.predict(X_test_transformed)
        print("\n=== Детальный отчет по Isomap + KNN Пайплайну ===")
        print(classification_report(y_test, y_pred_manifold, target_names=iris.target_names))

        # Научное обоснование результата
        delta = acc_baseline - acc_manifold
        print("=== Аналитический вывод ===")
        if delta > 0:
            print(f"Информационные потери составили {delta*100:.1f}%. Сжатие оправдано для целей визуализации.")
        else:
            print("Нелинейное многообразие Isomap успешно изолировало ключевые компоненты без потери точности.")

if __name__ == "__main__":
    benchmark = ManifoldInferenceBenchmark()
    benchmark.run_experiment()

[INFO] Исходная размерность пространства признаков: (120, 4)
[BENCHMARK] Точность на полном пространстве (4D): 1.0000

[INFO] Старт нелинейного сжатия пространства признаков до 2D...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_isomap.py:384: UserWarning: The number of connected components of the neighbors graph is 2 > 1. Completing the graph to fit Isomap might be slow. Increase the number of neighbors to avoid this issue.
  self._fit_transform(X)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


[BENCHMARK] Точность после Isomap сжатия (2D): 0.9333

=== Детальный отчет по Isomap + KNN Пайплайну ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

=== Аналитический вывод ===
Информационные потери составили 6.7%. Сжатие оправдано для целей визуализации.
